## Sarvam ASR Assignment: Diarization Benchmarking & Enhancement
---

**Deadline: Saturday, 22 August 2026, EOD**

You are given a CSV containing 100 YouTube videos with ground-truth diarization labels and reference transcripts. Your task is to parse these labels, benchmark existing diarization and ASR systems on them, and then **improve** the output.

### Input

`youtube_segments.csv` — one row per segment. The csv is present at https://drive.google.com/file/d/1Ijs1IWypIY2GAjpNUKV2XNZY6o7dFvSm/view?usp=sharing

Columns:

- `video_id` — YouTube video id
- `youtube_link` — source URL
- `start_sec`, `end_sec` — the window to extract. Use **only** `[start_sec, end_sec]`.
- `diarization_segments` — **ground-truth diarization**, formatted as `Speaker A [start-end] | Speaker B [start-end] | ...`
- `asr_segments` — **reference transcript** for the same segment, formatted as `[start-end] text | [start-end] text | ...`

`diarization_segments` and `asr_segments` share the **same segment boundaries in the same order** — the *n*-th `asr_segments` entry is the transcript of the *n*-th `diarization_segments` turn — so you can join speaker to text by index or by time. All timestamps are **relative to `start_sec`** (0 = start of the extracted clip).

If for any reason the ground-truth labels look incorrect, hand-label a subset and call out the limitation.

---

### Step 1 - Audio (or Video) Extraction

- Download the **audio** track from each YouTube link (yt-dlp / ffmpeg).
- Trim to `[start_sec, end_sec]`.
- Store as 16 kHz mono WAV.
- Optionally download the **video** track for the same window as well, if you plan to use visual cues in Step 4.

### Step 2 - Baseline Diarization Benchmarking

Run the extracted segments through top open-source diarization models.

Report standard metrics:

- **DER**
- **JER**
- Speaker-count accuracy

**Important:** Do NOT ignore overlapping speech regions when computing metrics. Overlap must be scored.

### Step 3 - ASR on Diarized Segments

Run ASR on the diarized speaker segments to produce speaker-attributed transcripts. Benchmark more than one STT system (e.g. Sarvam Saaras, Whisper, etc.) — the reference transcript in `asr_segments` is what you score against, not a substitute for running ASR yourself. Think about strategies for handling overlapping speech regions during ASR - how do you transcribe segments where multiple speakers are active simultaneously?

Also report:

- **cpWER**
- **WDER**

### Step 4 - Improving the Output

Take the best ASR + diarization combination you found while benchmarking, and build a pipeline on top of it that improves the output. Quantify the improvement over the same metrics as before. It is up to you whether you target diarization, ASR, or both.

**The ground truth is never an input to your pipeline.** `diarization_segments` and `asr_segments` are only for computing the final scores.

You are free to choose the strategy. Some directions:

- **Semantic / LLM-based correction** — feed the speaker-attributed text + timestamps to an LLM to detect and correct boundary errors, speaker confusion, and turn-merging issues. Any LLM is acceptable: paid APIs (GPT, Claude, Gemini, etc.) or open-source models. Prompting strategy, structured-output schema, chain-of-thought, and multi-pass refinement are yours to design.
- **Acoustic and visual cues** — use the video alongside the audio (e.g. active-speaker detection, face tracks, lip motion) to refine speaker boundaries and identities.
- Any combination of the above, or anything else you can justify.

**Hint:** Think about how transcript content can reveal diarization mistakes - e.g. unnaturally short segments, repeated/stuttered text across a speaker boundary suggesting a false split, or incoherent speaker transitions that indicate speaker confusion.

Also include:
- The papers / ideas your design draws from
- Multilingual / Indic-specific observations and failure modes

---

### Deliverables

Mail a Google Drive folder link (with all of the following):

1. A reproducible Colab notebook with the full pipeline end-to-end.
2. A results table: baseline vs. improved DER / JER / cpWER / WDER per model per video.
3. A short writeup (1–2 pages) covering approach, design choices, what worked, what didn't, failure cases, and references.

---

### Evaluation

- Correctness of pipeline and metrics
- Magnitude and consistency of improvement over baseline
- Depth of analysis and engineering judgement
- Clarity of code and writeup
- Breadth of techniques explored (bonus)


    

# Sarvam ASR Assignment — **Kaggle** runner

Identical pipeline to `main.ipynb`; only the platform glue differs. Before you
run anything, in the right-hand panel:

| Setting | Value | Why |
|---|---|---|
| **Internet** | **ON** | `pip install` and the `git clone` both need it. Off by default. |
| **Accelerator** | **GPU T4 x2** (or P100) | Step 2 inference |
| **Input** | your audio dataset | see 1.1 — `audio_16k/` and `meta/` |
| **Add-ons > Secrets** | `HF_TOKEN` | gated pyannote models (optional, see 1.1b) |

Storage differs from Colab: `/kaggle/input` is read-only and `/kaggle/working`
is writable but **wiped between sessions**. Cell 1.1 symlinks the dataset into
the working root so nothing is copied, and `Save Version` persists outputs.

## 1.0 — Setup

Installs `yt-dlp` (keep it current: YouTube extractor breakage is the single
most common cause of failure), mounts Drive, and puts `sarvam_diar/` on the
path. Everything here is cheap and safe to re-run.

In [ ]:
# --- dependencies -----------------------------------------------------------
# Kaggle: Settings (right panel) > Internet must be ON, or pip and the git clone
# below both fail. Accelerator > GPU T4 x2 (or P100) for Step 2.
#
# Installed HERE, before any work, because pyannote pulls a newer numpy than the
# image ships. Upgrading numpy under a live kernel leaves the process with
# half-old, half-new modules and the next numpy-touching import dies with
# `cannot import name '_center' from 'numpy._core.umath'`.
%pip install -q --upgrade yt-dlp
%pip install -q gdown "pyannote.audio>=4.0" "pyannote.metrics" itables

# Did an install replace a module this interpreter already imported? A fresh
# subprocess reads what is on disk; the running process reports what it loaded.
import os, shutil, subprocess, sys
from pathlib import Path

_stale = []
for _mod in ("numpy", "scipy"):
    try:
        _loaded = __import__(_mod).__version__
    except Exception:
        continue
    _disk = subprocess.run([sys.executable, "-c", f"import {_mod};print({_mod}.__version__)"],
                           capture_output=True, text=True).stdout.strip()
    if _disk and _disk != _loaded:
        _stale.append(f"{_mod}: loaded {_loaded}, on disk {_disk}")

if _stale:
    print("=" * 70)
    print("RESTART THE KERNEL, then run this cell again.")
    print("  Run > Restart & clear cell outputs")
    print("\nAn install replaced a package this session had already imported:")
    for _s in _stale:
        print("   ", _s)
    print("\nNothing is lost -- every stage is checkpointed under /kaggle/working,")
    print("so the cells below re-run in seconds after the restart.")
    print("=" * 70)
    raise SystemExit("restart required")

IN_KAGGLE = Path("/kaggle").exists()
print("running on Kaggle:", IN_KAGGLE)

# --- code -------------------------------------------------------------------
# Cloned fresh from GitHub so the code always matches a known revision. Needs
# Internet ON.
REPO_URL = "https://github.com/ParvGoyal08/MultilingualASR.git"
CODE_DIR = Path("/kaggle/working/sarvam-assignment" if IN_KAGGLE else "./sarvam-assignment")

def sync_code():
    if (CODE_DIR / ".git").exists():
        r = subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only", "-q"],
                           capture_output=True, text=True)
        if r.returncode:
            print("pull failed, re-cloning:", r.stderr.strip()[:200])
            shutil.rmtree(CODE_DIR, ignore_errors=True)
        else:
            return CODE_DIR
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE_DIR)], check=True)
    return CODE_DIR

PROJECT_DIR = sync_code()
sha = subprocess.run(["git", "-C", str(PROJECT_DIR), "log", "-1", "--format=%h  %s"],
                     capture_output=True, text=True).stdout.strip()
print(f"code @ {sha}")

sys.path.insert(0, str(PROJECT_DIR))

# Drop any previously-imported sarvam_diar modules so the freshly pulled code is
# what actually gets imported.
for _m in [k for k in list(sys.modules)
           if k == "sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[_m]

try:
    from IPython import get_ipython

    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload: on")
except Exception as exc:
    print(f"autoreload unavailable ({type(exc).__name__}) -- re-run this cell "
          "after a push to pick up new code")

import pandas as pd

import sarvam_diar
from sarvam_diar import analysis, config, data, extraction, reference, utils
print("sarvam_diar", sarvam_diar.__version__)

## 1.1 — Config and stage flags

One visible cell holding every path and toggle. Each stage checkpoints to Drive,
so the normal workflow is: enable one stage, let it finish, flip it off, move
on. Re-running the notebook top-to-bottom then costs nothing but a few file
reads — which is what makes it survivable on a runtime that disconnects every
90 minutes.

In [ ]:
from sarvam_diar.config import Config, StageFlags

# Kaggle splits storage in two: /kaggle/input is READ-ONLY (your uploaded
# dataset) and /kaggle/working is writable but wiped between sessions. The
# pipeline wants one root, so audio_16k/ and meta/ are symlinked from the
# dataset into the working root -- no 1.3 GB copy, and every stage still writes
# its checkpoints normally.
ROOT = Path("/kaggle/working/sarvam_diarization") if IN_KAGGLE else Path("./pipeline_out")
ROOT.mkdir(parents=True, exist_ok=True)

# Set this to your dataset directory. Check the right-hand Input panel for the
# exact name -- Kaggle slugifies it (e.g. "Sarvam Audio" -> sarvam-audio).
DATASET = Path("/kaggle/input/sarvam-diarization-audio")

if IN_KAGGLE:
    if not DATASET.exists():
        candidates = sorted(p.name for p in Path("/kaggle/input").glob("*"))
        raise FileNotFoundError(
            f"{DATASET} not found. Datasets currently attached: {candidates or 'none'}.\n"
            "Add your dataset via the right panel (Input > Add Input), then set "
            "DATASET above to its directory name."
        )
    for sub in ("audio_16k", "meta"):
        src, dst = DATASET / sub, ROOT / sub
        if dst.is_symlink() or dst.exists():
            continue
        if src.exists():
            dst.symlink_to(src)          # read-only is fine: nothing rewrites these
            print(f"linked {sub} -> {src}")
        else:
            print(f"WARNING: {src} missing in the dataset")

cfg = Config.create(
    root=ROOT,
    work_dir=Path("/kaggle/working/tmp") if IN_KAGGLE else Path("./.work"),
    # force_client="android",   # pin a yt-dlp player client (extraction only)
)

flags = StageFlags(
    run_extraction=False,      # Kaggle IPs are bot-gated by YouTube just like
                               # hosted runtimes -- extract locally and upload the audio.
    build_reference=True,      # rebuilt from the CSV, cheap
    run_diarization=True,      # Step 2 -- gates BOTH the smoke run (2.1)
                               #           and the full sweep (2.2)
    run_oracle_count_ablation=False,
    run_asr=False,             # Step 3 (still a stub)
    run_refinement=False,      # Step 4 (still a stub)

    force_redo=False,          # re-run enabled stages from scratch, ignoring
                               # checkpoints. Honoured by extraction, reference
                               # AND diarization. No need to delete anything.
    retry_failed_only=False,
    limit=None,
    only_clip_ids=None,
)

print(cfg.describe())
print(flags.describe())

### 1.1b — One-time: credentials

The gated pyannote models need a HuggingFace token. Two ways on Kaggle, in
order of preference:

1. **Add-ons > Secrets** (right panel) — add `HF_TOKEN`, then tick it for this
   notebook. It persists across sessions and is never written to disk. The
   pipeline picks it up automatically; you can skip the cell below.
2. **The cell below** — writes `/kaggle/working/sarvam_diarization/.env` with
   `getpass`, so nothing is echoed or saved into the notebook. Convenient, but
   `/kaggle/working` is wiped between sessions, so you would repeat it.

Either way the token never appears in a cell, which matters because this
notebook is committed to a public repo.

In [ ]:
# Skip this entirely if you added HF_TOKEN under Add-ons > Secrets.
from getpass import getpass
from sarvam_diar import diarization as _dz

if _dz.resolve_token(cfg):
    print("token already available (Kaggle Secrets or an existing .env) -- nothing to do")
else:
    print("target:", cfg.dotenv_path)
    for key, why in (("HF_TOKEN", "gated pyannote models, Step 2"),
                     ("SARVAM_API_KEY", "Sarvam ASR, Step 3")):
        value = getpass(f"{key} ({why}) -- blank to skip: ").strip()
        if value:
            utils.set_dotenv_value(cfg.dotenv_path, key, value)
            print(f"  {key} written ({len(value)} chars)")
        else:
            print(f"  {key} skipped")

## 1.2 — Load the CSV and parse the ground truth

The CSV is fetched to Drive once and reused (skip-if-exists). Parsing asserts
that the two label columns agree segment-for-segment, and drops the two corrupt
segments documented in Step 0 with an explicit record of what was excluded.

In [ ]:
df = data.load_segments_csv(cfg)
clips = data.parse_ground_truth(df)

print(f"{len(df)} rows, {df.video_id.nunique()} unique video_ids, "
      f"{sum(len(c.segments) for c in clips)} segments")

# The duplicated video_id is why clip_id is composite, not just video_id.
dupes = df[df.video_id.duplicated(keep=False)]
print("\nSame video, two windows -> two distinct clip_ids:")
display(dupes[["video_id", "start_sec", "end_sec", "clip_id"]])

gt = data.clips_to_frame(clips)
display(gt.head(3))

## 1.3 — Dataset profile

Recomputes every number quoted in Step 0 from the file itself and writes
`results/dataset_profile.json`. Run it after any change to the parser: if a
number here moves, the Step 0 notes are stale.

In [ ]:
import json
profile = data.profile_dataset(clips, cfg)

print("clips              ", profile["n_rows"], "rows /", profile["n_unique_video_ids"], "videos")
print("segments           ", profile["n_segments"], f"({profile['n_dropped_segments']} dropped as malformed)")
print("audio              ", profile["clip_duration_sec"]["total_hours"], "h")
print("speakers per clip  ", profile["speakers"]["per_clip_distribution"])
print("overlap            ", f"{profile['overlap']['overlap_frac_of_corpus']:.2%} of corpus time,",
      profile["overlap"]["n_clips_without_overlap"], "clips with none")
print("GT beyond end_sec  ", profile["gt_boundary"]["n_clips_gt_beyond_end_sec"], "clips,",
      f"median +{profile['gt_boundary']['overrun_median_sec']}s,",
      f"max +{profile['gt_boundary']['overrun_max_sec']}s")
print("scripts            ", profile["language"]["script_distribution"])
print("non-speech tags    ", profile["transcript"]["nonspeech_tags"])
print("code-switch glosses", profile["transcript"]["n_code_switch_glosses"],
      "in", profile["transcript"]["n_segments_with_gloss"], "segments")

print("\nDropped ground-truth segments:")
print(json.dumps(profile["dropped_segments"], indent=2, ensure_ascii=False))

### 1.3b — What is actually present under `cfg.root`?

Step 1 may have run on a different machine (it did here: extraction locally,
because the hosted IPs are bot-gated by YouTube). The artifacts arrive by a
manual copy, which can be partial — so check before depending on them, rather
than discovering it as a traceback three cells later.

In [ ]:
inv = extraction.inventory(cfg, expected_clips=len(clips))
display(inv[["artifact", "present", "count", "expected"]])

missing = inv[~inv.present]
if len(missing):
    print("MISSING:")
    for a in missing.artifact:
        print("   ", a)
    print(f"\nCopy the CONTENTS of your local `local_out/` folder into {cfg.root}\n"
          "so it holds audio_16k/, meta/, results/, reference/ and data/.")
else:
    print("all Step 1 artifacts present")

# Exactly which clips lack usable audio, largest first. A manual copy that
# stopped part-way shows up here as the biggest files -- re-upload just these
# rather than the whole 1.3 GB.
gaps = extraction.missing_audio(cfg, clips)
if len(gaps):
    print(f"\n{len(gaps)} clip(s) without usable audio, "
          f"{gaps.expect_bytes.sum()/1e6:.0f} MB to transfer:")
    display(gaps[["clip_id", "state", "expect_mb", "duration_sec"]])
    print("GUVrL5ltiP4 is expected here -- that source video is deleted.")
    print("For anything else: copy those WAVs into", cfg.audio_dir)
    print("and keep run_extraction=False -- Kaggle IPs are bot-gated too.")
else:
    print("\nevery clip has a usable WAV")

# 99 not 100 is expected: GUVrL5ltiP4 is a deleted video.
short = inv[(inv["count"] < inv.expected) & inv.present]
if len(short):
    print("\nfewer than expected (99/100 is normal -- one source video is deleted):")
    display(short[["artifact", "count", "expected"]])

## 1.4 — Extract

Per clip: resolve the media URL, range-fetch only `[start_sec, end_sec]` with
ffmpeg, force the exact sample count, publish atomically to Drive, then write
the sidecar as the commit marker.

**Fetch strategy.** Which YouTube player client yields a *downloadable* format
is not stable — it varies by network, video and yt-dlp version, and the client
with the best formats is often not a working one. Measured while building this:
yt-dlp's default rotation offers opus audio-only (itag 251) but those URLs 403
without a PO token, while `android` offers only itag 18 (muxed 360p) and
downloads fine. So the ladder is *walked*, not guessed, and whichever client
works is reused for the remaining clips — only the first clip pays for the
search. Tier A range-fetches just the window; tier B downloads the full track
and trims locally, and is used when tier A fails or its output drifts beyond
`DUR_TOL_SEC`. Both were verified to produce **byte-identical** audio.

Safe to interrupt and re-run — finished clips are skipped. Kaggle's IPs are bot-gated by YouTube exactly as Colab's are, so extraction
is expected to run locally; keep `run_extraction=False` here.

In [ ]:
if flags.run_extraction:
    results = extraction.run(cfg, clips, flags)
else:
    # The per-clip sidecars in meta/ are the commit markers and the authority;
    # step1_extraction.csv is derived. If they disagree -- e.g. a stale CSV from
    # an earlier failed run survived while the audio was copied in from another
    # machine -- trust the sidecars and repair the CSV.
    results, recon = extraction.reconcile(cfg, clips)
    print(f"extraction skipped. {recon['action']}: "
          f"csv={recon['csv_ok_rows']} ok, sidecars={recon['sidecar_ok_rows']} ok")
    if recon.get("note"):
        print("\n" + recon["note"])
    if int((results.status == "ok").sum()) == 0:
        raise FileNotFoundError(
            f"No successfully extracted clips under {cfg.root}.\n"
            "Neither the results CSV nor the per-clip sidecars in meta/ show any.\n"
            "Copy the CONTENTS of your local `local_out/` folder into that "
            "directory (see the inventory cell above)."
        )

## 1.5 — Report

Per-video results, the run summary, and the failure breakdown. `all_exact` is
the one that matters: it asserts every published WAV holds exactly the requested
number of samples.

In [ ]:
import json

# The CSV is the source of truth; the summary JSON is optional enrichment. If
# Step 1 ran elsewhere and only part of results/ was copied across, report from
# what is here instead of failing.
summary = utils.read_json(cfg.extraction_summary)
if summary:
    print(json.dumps({k: summary[k] for k in
                      ("totals", "audio_contract", "download_tiers", "player_clients",
                       "formats", "error_classes", "session_counts") if k in summary},
                     indent=2, ensure_ascii=False))
    assert summary["audio_contract"]["all_exact"], "some WAVs are not exactly the requested length"
else:
    print(f"note: {cfg.extraction_summary.name} not found -- reporting from "
          f"{cfg.extraction_csv.name} instead.\n")

done = results[results.status == "ok"]
hours = pd.to_numeric(done.requested_dur_sec, errors="coerce").sum() / 3600
gb = pd.to_numeric(done.file_size_bytes, errors="coerce").sum() / 1e9
print(f"{len(done)} extracted / {len(results)} rows | {hours:.2f} h | {gb:.2f} GB")

# Re-derive the exact-length contract from the CSV, so it is checked even when
# the summary JSON is absent.
n = pd.to_numeric(done.n_samples, errors="coerce")
exp = pd.to_numeric(done.n_expected_samples, errors="coerce")
print(f"exact-length contract: {int((n == exp).sum())}/{len(done)} rows exact")
assert (n == exp).all(), "some WAVs are not exactly the requested sample count"

display(done[["clip_id", "requested_dur_sec", "n_samples", "n_expected_samples",
              "raw_delta_sec", "pad_samples", "trim_samples", "download_tier",
              "player_client", "ytdlp_format_id", "attempts", "elapsed_sec",
              "n_gt_speakers", "gt_overlap_frac", "ref_lang_hint"]].head(20))

failed = results[results.status == "failed"]
if len(failed):
    print(f"\n{len(failed)} failed ({int(failed.permanent.sum())} permanent):")
    display(failed[["clip_id", "error_class", "permanent", "attempts", "error_msg"]])
else:
    print("\nno failures")

attempts_log = pd.DataFrame(utils.read_jsonl(cfg.failures_jsonl))
if len(attempts_log):
    print(f"\nfailed attempts by class and client ({len(attempts_log)} total):")
    display(attempts_log.groupby(["error_class", "player_client"]).size()
            .rename("n").reset_index())

## 1.6 — Verification

Three checks, in order of how much they would cost to get wrong:

1. **Audit** — re-probes every published WAV against the exact-length contract
   and reports orphans, so a corrupted checkpoint is caught before Step 2 reads
   it as truth.
2. **Alignment** — the important one. Ground-truth timestamps are relative to
   `start_sec`, so a one-second trim offset would silently wreck DER for that
   clip with no visible symptom. Slices the first few ground-truth segments out
   of a published WAV and renders them next to their reference text: if the
   offset is wrong, the audio will not match the words.
3. **Idempotency** — re-running the extract cell must skip everything and touch
   the network zero times.

In [ ]:
# --- 1. audit every published WAV -------------------------------------------
audit = extraction.audit(cfg, clips)
attempted = audit[audit.clip_id.isin(results[results.status == "ok"].clip_id)]
print(f"audit: {attempted.valid.sum()}/{len(attempted)} extracted clips pass the exact-length contract")
bad = attempted[~attempted.valid]
if len(bad):
    display(bad)

orphans = {p.stem for p in cfg.audio_dir.glob("*.wav")} - {c.clip_id for c in clips}
print("orphan WAVs on Drive:", sorted(orphans) or "none")

In [ ]:
# --- 2. alignment spot-check ------------------------------------------------
# If the trim offset were wrong, the words below would not match the audio.
from IPython.display import Audio, display as ipy_display

ok_ids = results[results.status == "ok"].clip_id.tolist()
if not ok_ids:
    print("no successfully extracted clips -- see the inventory cell (1.3b)")
    print("status counts:", results.status.value_counts().to_dict())
else:
    clip = next(c for c in clips if c.clip_id == ok_ids[0])
    wav = cfg.wav_path(clip.clip_id)     # resolved against THIS machine's cfg
    if not wav.exists():
        print(f"results say {clip.clip_id} is extracted, but {wav} is missing.\n"
              "The audio_16k/ folder did not come across -- see cell 1.3b.")
    else:
        print(f"{clip.clip_id}  |  {clip.stats['lang_script']}  |  "
              f"{clip.stats['n_gt_speakers']} speakers  |  {len(clip.segments)} segments")
        for seg in clip.segments[:4]:
            print(f"\n[{seg.start:7.2f} - {seg.end:7.2f}]  {seg.speaker}")
            print(f"   {seg.text[:160]}")
            ipy_display(Audio(extraction.read_wav_window(wav, seg.start, seg.end),
                              rate=cfg.sample_rate))

In [ ]:
# --- 3. idempotency ---------------------------------------------------------
# Must report every already-extracted clip as skipped, with no network traffic.
_ = extraction.run(cfg, clips, StageFlags(only_clip_ids=ok_ids[:3]))

---

# Step 1.7 — Build the scoring reference

Everything so far has been pipeline-side. This section builds the **scoring
reference** from the ground truth, and puts a hard wall between the two.

The brief is unambiguous:

> **The ground truth is never an input to your pipeline.** `diarization_segments`
> and `asr_segments` are only for computing the final scores.

### Why the reference has to be built rather than scored raw

| Problem | Scale | Consequence if ignored |
|---|---|---|
| GT runs past the audio window | 85/100 clips, median +1.43 s | hypothesis silence scored against speech that is not in the file |
| Same-speaker intervals overlap themselves | 140 pairs | one speaker's second counted twice in the DER denominator |
| Corrupt segments | 2 (`end < start`, zero-length) | undefined turn geometry |
| **Code-switch gloss is not speech** | **24,914 of 149,121 tokens — 16.7%** | **a ~17% cpWER floor that says nothing about ASR quality** |

### What is deliberately *not* done

No overlap removal, no forgiveness collar as the headline metric, no
minimum-duration filter, no per-system normalization. Overlap in particular is
scored in full — the brief requires it.

### The line between normalization and cheating

The normalizer is legitimate because it is (1) a pure function of one text
string with no access to the other side, (2) applied **identically** to
reference and hypothesis, (3) fixed and versioned before any system output was
seen, and (4) documented with its corpus-wide token impact. `strip_gloss` is the
only asymmetry and it is inert — hypotheses contain no glosses, which the
verification cell asserts.

In [ ]:
# Ground truth in -> ClipInput (pipeline-safe) + ClipReference (scoring only).
inputs, ref_clips = data.split_reference(clips, results, cfg=cfg)
print(f"{len(inputs)} pipeline inputs (extracted clips only), {len(ref_clips)} reference clips")
print("ClipInput fields:", sorted(data.ClipInput.__dataclass_fields__))

if flags.build_reference:
    manifest = reference.run(cfg, clips, force=flags.force_redo)
else:
    manifest = pd.read_csv(cfg.reference_manifest)

display(manifest[["clip_id", "n_turns", "n_speakers", "speaker_time_sec", "overlap_sec",
                  "overlap_frac", "n_utterances", "n_ref_tokens",
                  "n_segments_dropped_by_uem", "n_segments_truncated_by_uem",
                  "n_same_speaker_merges"]].head(10))

print(f"\nturns {int(manifest.n_turns.sum())} | utterances {int(manifest.n_utterances.sum())} "
      f"| reference tokens {int(manifest.n_ref_tokens.sum())}")
print(f"UEM crop: {int(manifest.n_segments_dropped_by_uem.sum())} segments dropped, "
      f"{int(manifest.n_segments_truncated_by_uem.sum())} truncated, "
      f"{int(manifest.n_same_speaker_merges.sum())} same-speaker merges")
print("reference speaker counts:", manifest.n_speakers.value_counts().sort_index().to_dict())

In [ ]:
# --- normalization report + the golden table ---------------------------------
import json
report = reference.normalization_report(clips, cfg)
print(json.dumps(report, indent=2, ensure_ascii=False))

print("\nGolden cases -- every hard variant found in the corpus, eyeball these:\n")
for case in reference.GOLDEN_CASES:
    print("RAW :", case)
    print("NORM:", reference.normalize_text(case) or "(empty -- non-speech, excluded from WER)")
    print()

## 1.8 — Reference and leak-guard verification

Fifteen assertions. The ones that matter most:

* **Cropping must not change the speaker set** — otherwise speaker-count
  accuracy is measured against a reference the audio cannot support.
* **Cross-speaker overlap must survive at 7.13%** — a regression here means the
  reference quietly stopped scoring the overlap the brief requires.
* **The normalizer must be idempotent** and must never empty a speech segment.
* **`ClipInput` must carry no ground-truth field**, and the DataFrame guard must
  reject `n_gt_speakers` / `ref_lang_hint`. That column pair is exactly what
  would otherwise become pyannote's `num_speakers` or Whisper's `language`.

In [ ]:
import re, collections
refs = {c.clip_id: reference.build_reference(c) for c in clips}
speech = [s for c in clips for s in c.segments if s.is_speech]
failures = []

def check(name, ok, detail=""):
    print(f"  {'PASS' if ok else 'FAIL'}  {name}{'  ' + detail if detail else ''}")
    if not ok:
        failures.append(name)

print("=== reference integrity ===")
check("no turn outside the UEM",
      not [t for r in refs.values() for t in r.turns if t.end > r.uem[1] + 1e-6 or t.start < -1e-9])
# Compare through rttm_safe: reference turn labels are normalised to the
# RTTM-legal form (`Speaker A` -> `Speaker_A`) at build time, while raw segments
# keep the CSV spelling. The property under test is that cropping loses no
# speaker, not how the label is spelled.
check("cropping preserves the speaker set",
      all({reference.rttm_safe(s.speaker) for s in c.segments}
          == {t.speaker for t in refs[c.clip_id].turns} for c in clips),
      f"{len(clips)}/{len(clips)}")
disjoint = True
for r in refs.values():
    per = collections.defaultdict(list)
    for t in r.turns:
        per[t.speaker].append((t.start, t.end))
    for v in per.values():
        v.sort()
        disjoint &= all(v[i][0] >= v[i - 1][1] - 1e-9 for i in range(1, len(v)))
check("same-speaker intervals disjoint after union", disjoint)
frac = sum(r.stats["overlap_sec"] for r in refs.values()) / sum(c.duration for c in clips)
check("cross-speaker overlap preserved", abs(frac - 0.0713) < 0.002, f"{frac:.4%} of corpus")
check("RTTM round-trips exactly",
      all(reference.parse_rttm(reference.to_rttm(r)) ==
          [data.Turn(t.speaker, round(t.start, 6), round(t.end, 6)) for t in r.turns]
          for r in refs.values()))

print("\n=== normalizer ===")
norm = [reference.normalize_text(s.text) for s in speech]
check("idempotent", all(reference.normalize_text(n) == n for n in norm), f"{len(speech)} segments")
check("no speech segment normalizes to empty", all(norm))
check("token count matches the report",
      sum(len(n.split()) for n in norm) == report["tokens"]["gloss_stripped_final"],
      f"{sum(len(n.split()) for n in norm)}")
check("reference is essentially pure native script",
      len(re.findall(r'[a-z]+', ' '.join(norm))) <= 40,
      f"{len(re.findall(r'[a-z]+', ' '.join(norm)))} residual Latin tokens")
gloss_free = ["नमस्कार मी गौरव जोशी आहे", "இது ஒரு சோதனை", "hello this is a test"]
check("gloss step is inert on gloss-free text (ref/hyp symmetry)",
      all(reference.normalize_text(h, True) == reference.normalize_text(h, False) for h in gloss_free))

print("\n=== leak guard ===")
fields = set(data.ClipInput.__dataclass_fields__)
check("ClipInput has no segments attribute", "segments" not in fields)
check("ClipInput carries no gt_/ref_ field", not utils.reference_fields(fields))
for bad in ("n_gt_speakers", "ref_lang_hint"):
    try:
        utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", bad]))
        check(f"guard rejects {bad}", False)
    except AssertionError:
        check(f"guard rejects {bad}", True)
utils.assert_no_reference_fields(pd.DataFrame(columns=["clip_id", "wav_path"]))
check("guard passes clean columns", True)
check("split_reference yields ClipInput only",
      all(isinstance(v, data.ClipInput) for v in inputs.values()), f"{len(inputs)} inputs")

assert not failures, f"reference verification failed: {failures}"
print("\nALL CHECKS PASSED")

---

# Step 2 — Baseline diarization benchmarking

Two pyannote pipelines over the extracted clips: **community-1** (primary) and
**speaker-diarization-3.1** (the baseline it replaced). Same library, one code
path, and the delta between them is the version-over-version comparison.

### Scoring rules, and why

The brief is explicit: *"Do NOT ignore overlapping speech regions when computing
metrics. Overlap must be scored."* So the headline numbers use **collar 0.0 and
`skip_overlap=False`** — score everything, forgive nothing. That matters here
more than usual: **7.13% of scored time has ≥2 speakers active** and only 9 of
100 clips have none. A collared, overlap-forgiving variant is reported
alongside for comparability with published figures, never instead.

### No speaker-count hints

`num_speakers` / `min_speakers` / `max_speakers` are never passed. The pipeline
has to estimate the count itself — that estimate is exactly what speaker-count
accuracy measures, and feeding it the reference count is the classic leak.
`data.ClipInput` makes this structural: the type handed to `diarization.run()`
has no speaker field to leak.

### Credentials

Both pipelines are gated. The token is read from **`.env`**, never from a
notebook cell. `.env` is gitignored so it never reaches the public repo — which
also means it does not arrive with the git clone, so on Kaggle use Add-ons > Secrets at `/kaggle/working/sarvam_diarization/.env`:

```
HF_TOKEN="hf_..."
SARVAM_API_KEY="..."      # Step 3
```

### What to expect

community-1 reports ~11–20% DER on standard benchmarks. **Ours will be worse.**
This is 9-language Indic code-switched YouTube audio, 20% of reference segments
are under 0.5 s, and we score with no collar and full overlap. **25–40% DER
would be unsurprising and is not a bug.** A DER near 0, or above ~70%, probably
is.

In [ ]:
# pyannote was installed in cell 1.0, deliberately: installing it here would
# upgrade numpy under a live kernel and break the session. Nothing is installed
# in this cell -- it only verifies.
#
# pip's resolver warnings during 1.0 (numba wanting numpy<2.1, google-adk wanting
# older opentelemetry) are about image preinstalls, not our dependencies:
# pyannote.audio declares no numba/librosa dependency and importing the pipeline
# pulls in neither. This check is what actually decides usability.
import sys, importlib

for mod in ("torch", "pyannote.audio", "pyannote.metrics", "pyannote.core"):
    m = importlib.import_module(mod)
    print(f"  {mod:18s} {getattr(m, '__version__', '?')}")

from pyannote.audio import Pipeline                      # noqa: F401
from pyannote.audio.pipelines import SpeakerDiarization  # noqa: F401
from pyannote.metrics.diarization import DiarizationErrorRate  # noqa: F401
leaked = sorted(m for m in sys.modules if m.split(".")[0] in ("numba", "llvmlite"))
print(f"  numba pulled in by pyannote: {leaked or 'no'}")
print("  imports OK -- resolver warnings above do not affect this pipeline\n")

from sarvam_diar import diarization, evaluation

# --- GPU ---------------------------------------------------------------------
dev = diarization.device_report()
print("device:", dev)
if not dev["cuda"]:
    print("\nWARNING: no GPU. Settings (right panel) > Accelerator > GPU T4 x2, then re-run.")

# --- HF token from .env -------------------------------------------------------
# community-1 and 3.1 are gated. The token is read from a .env file, never from
# a cell -- this notebook is published to a public repo.
#
# .env is gitignored, so it does NOT arrive with the git clone. On Kaggle use
# Add-ons > Secrets, or cell 1.1b. Same file also carries SARVAM_API_KEY for
# Step 3.
tok = diarization.resolve_token(cfg)
print("expected .env:", cfg.dotenv_path)
# Only ever print a masked form: this output is saved into the .ipynb.
print("HF token     :", f"found ({tok[:5]}...{tok[-3:]}, {len(tok)} chars)" if tok
      else f"MISSING -- run cell 1.1b to write it to {cfg.dotenv_path}")
if tok:
    print("               you must also accept conditions at:")
    for k, repo in config.DIARIZATION_MODELS.items():
        print(f"                 https://hf.co/{repo}")

# --- metric semantics probe ---------------------------------------------------
# Re-derives pyannote.metrics' behaviour at runtime rather than trusting the
# docs, which pin down none of it. If a library upgrade renames a component or
# switches accumulation from pooling to averaging, this catches it here instead
# of silently shifting every reported number.
probe = evaluation.probe_metric_semantics()
for k, v in probe.items():
    print(f"  {k}: {v}")
assert not probe["problems"], probe["problems"]

## 2.1 — Smoke run: measure throughput before committing to the full sweep

Three clips chosen to span the duration range — RTF is not constant, so a smoke
set of three short clips would give a misleadingly optimistic projection. Fixed
per-call overhead dominates on short clips; memory pressure shows up on long
ones, and the longest clip in the corpus is 30 minutes.

The projection this prints is the gate on whether to run the full sweep.

In [ ]:
# ClipInput only -- no speaker counts, no language, no ground truth.
inputs, ref_clips = data.split_reference(clips, results, cfg=cfg)
by_id = {c.clip_id: c for c in inputs.values()}
print(f"{len(inputs)} extracted clips available to the pipeline")

# short / mid / long, so the RTF spread is visible
ordered = sorted(inputs.values(), key=lambda c: c.duration)
smoke = [ordered[0], ordered[len(ordered) // 2], ordered[-1]]
for c in smoke:
    print(f"  smoke: {c.clip_id:<28} {c.duration:7.0f}s")

if not flags.run_diarization:
    raise SystemExit("flags.run_diarization is False -- set it True in cell 1.1")

smoke_df = diarization.run(cfg, smoke,
                           StageFlags(force_redo=flags.force_redo))
display(smoke_df[["model", "clip_id", "status", "n_turns", "n_speakers_hyp",
                  "elapsed_sec", "rtf"]])

tput = diarization.throughput_report(smoke_df)
print("\nMEASURED throughput (not an estimate):")
display(tput)
print(f"GPU: {dev['device_name']}")
print("\n^ projected_full_sweep is per model, for all 12.26 h. Decide from this "
      "whether to run the full sweep now, trim the roster, or change runtime tier.")

## 2.2 — Full sweep

Checkpointed per clip: RTTM written first, sidecar JSON last as the commit
marker. Safe to interrupt — finished clips are skipped and a session restart
costs only the clip in flight.

Set `LIMIT = None` to run everything.

In [ ]:
LIMIT = None          # e.g. 10 to extend gradually; None runs all clips

if not flags.run_diarization:
    print("=" * 68)
    print("FULL SWEEP SKIPPED -- flags.run_diarization is False.")
    print("Only the 3 smoke clips from 2.1 have been diarized, and LIMIT is")
    print("not even consulted. Set run_diarization=True in cell 1.1, re-run it,")
    print("then re-run this cell.")
    print("=" * 68)
else:
    n_clips = len(inputs) if LIMIT is None else min(LIMIT, len(inputs))
    print(f"sweeping {n_clips} clips x {len(config.DIARIZATION_MODELS)} models "
          f"= {n_clips * len(config.DIARIZATION_MODELS)} runs "
          f"(already-finished clips are skipped)\n")

    hyp_df = diarization.run(cfg, list(inputs.values()),
                             StageFlags(limit=LIMIT, force_redo=flags.force_redo))
    ok = hyp_df[hyp_df.status == "ok"]
    print(f"\n{len(ok)} hypotheses / {len(hyp_df)} attempts")

    # Did the sweep actually cover everything it was asked to?
    expected = n_clips * len(config.DIARIZATION_MODELS)
    if len(ok) < expected:
        print(f"WARNING: expected {expected} successful runs, got {len(ok)}. "
              "Check the failures below and the LIMIT above.")
    if (hyp_df.status == "failed").any():
        display(hyp_df[hyp_df.status == "failed"][["model", "clip_id", "error_class"]])

## 2.3 — Score: DER, JER, speaker-count accuracy

Corpus DER is computed by **pooling seconds**, not by averaging per-clip DER.
Averaging would weight a 50-second clip the same as a 30-minute one — a
different, and wrong, statistic. The probe in 2.0 confirmed pyannote pools too,
so both routes agree.

In [ ]:
# Load reference + hypotheses from the checkpoints, then score.
references = {cid: reference.load_reference(cfg, cid) for cid in inputs}
hypotheses = {}
for model in config.DIARIZATION_MODELS:
    for cid in inputs:
        if diarization.is_done(cfg, model, cid):
            hypotheses[(model, cid)] = diarization.load_hypothesis(cfg, model, cid)
print(f"scoring {len(hypotheses)} (model, clip) pairs")

metrics = evaluation.score_all(cfg, references, hypotheses)
corpus = evaluation.aggregate(metrics)

if not len(metrics):
    print("\nNothing to score yet -- no hypotheses on disk.")
    print("Run cells 2.1 / 2.2 first. If they failed with GatedRepoError, accept")
    print("the model conditions at the HuggingFace links printed in cell 2.0")
    print("(including pyannote/segmentation-3.0, which 3.1 pulls in).")
else:
    print("\n=== CORPUS (collar 0.0, overlap scored) ===")
    display(corpus[["model", "n_clips", "der", "der_fa_frac", "der_miss_frac",
                    "der_confusion_frac", "jer_mean", "speaker_count_accuracy",
                    "speaker_count_mae", "speaker_count_bias"]])

    print("=== lenient variant (collar 0.25, overlap skipped) -- secondary only ===")
    display(corpus[["model", "lenient_der", "lenient_jer_mean"]])

    print("=== per-clip (head) ===")
    display(metrics[["model", "clip_id", "der", "jer", "n_speakers_ref",
                     "n_speakers_hyp", "speaker_count_error"]].head(12))

## 2.4 — Where does the error actually come from?

DER alone does not say whether the system got the **number** of speakers wrong,
or got the number right and assigned the wrong one. Two analyses:

**(a) Stratified — leak-free.** Clips split by `n_hyp − n_ref`. Inside the
`exact` stratum no reference speaker is unmappable, so its confusion component
is *pure assignment error*. Over-estimation shows up as confusion plus false
alarm; under-estimation as missed detection plus confusion.

**(b) Oracle-count ablation — a diagnostic, quarantined.** Re-runs with
`num_speakers` set from the reference. The DER gap is what count estimation
costs, in DER points. This *does* feed ground truth to the model, so it is
off by default, writes to a separate `hypotheses_oracle/` tree, prefixes every
column `oracle_`, and must never be reported as system performance.

In [ ]:
if not len(metrics):
    print("no metrics yet -- see cell 2.3")
else:
    print("=== DER by speaker-count stratum (leak-free) ===")
    strata = evaluation.stratify_by_count_error(metrics)
    display(strata[["model", "stratum", "n_clips", "der", "der_fa_frac",
                    "der_miss_frac", "der_confusion_frac"]])
    print("In the `exact` stratum, der_confusion_frac is pure assignment error.\n")

    print("=== count error vs DER ===")
    display(evaluation.count_error_correlation(metrics))

    # pooling the strata must reproduce the headline number
    for model, sub in metrics.groupby("model"):
        pooled = evaluation.pool(sub.to_dict("records"))["der"]
        head = corpus.loc[corpus.model == model, "der"].iloc[0]
        assert abs(pooled - head) < 1e-9, f"{model}: strata do not reconcile"
    print("strata reconcile with the corpus DER\n")

    # --- oracle ablation, opt-in, DIAGNOSTIC ONLY ---------------------------
    if flags.run_oracle_count_ablation:
        oracle_counts = {cid: references[cid].n_speakers for cid in inputs}
        oracle_hyp_df = diarization.run(cfg, list(inputs.values()),
                                        StageFlags(limit=LIMIT, force_redo=flags.force_redo),
                                        oracle_counts=oracle_counts)
        oracle_hyps = {(m, cid): diarization.load_hypothesis(cfg, m, cid, oracle=True)
                       for m in config.DIARIZATION_MODELS for cid in inputs
                       if diarization.is_done(cfg, m, cid, oracle=True)}
        oracle_metrics = evaluation.score_all(None, references, oracle_hyps)
        if len(oracle_metrics):
            print("=== ORACLE ABLATION -- diagnostic, NOT system performance ===")
            display(evaluation.oracle_gap(metrics, oracle_metrics))
    else:
        print("oracle ablation off (flags.run_oracle_count_ablation=False)")

    summary = evaluation.summarize(cfg, metrics)
    print(f"\nwrote {cfg.step2_metrics_csv}")
    print(f"wrote {cfg.step2_summary}")

## 2.5 — Error analysis: where the DER actually comes from

Eight rankings plus a head-to-head. Tables are interactive (sortable, searchable,
paginated) — click a column header to re-sort.

**Rate and contribution answer different questions.** A 50 s clip at DER 0.90 has
a terrible rate but is a rounding error in the corpus number; a 30-minute clip at
DER 0.30 can be a fifth of all the error in the benchmark. Every table below
carries both `der` (the rate) and `error_share` (this clip's share of the model's
total error seconds). **"Biggest contributors" is the list to act on** — those are
the clips where an improvement actually moves the headline.

Reading the error types:

| column | means | usually caused by |
|---|---|---|
| `der_miss_sec` | reference speech never detected | VAD/segmentation; overlapped and nested speech |
| `der_confusion_sec` | speech found, wrong speaker | clustering — too few/many clusters, or bad embeddings |
| `der_fa_sec` | speech emitted where reference has none | over-eager VAD — **or unannotated reference tails** |
| `overlap_der` | DER scored *only* where ≥2 speakers are active | the hardest condition; NaN where a clip has no overlap |

On false alarm specifically: `Cku_X_SL7qU` (90 s unannotated), `OxYCBQKZ3iY` (12 s)
and `CO_8ppdzq9U` (6 s) have tails the annotator never labelled, so a high FA
there may be the reference's fault rather than the model's.

In [ ]:
if not len(metrics):
    print("no metrics yet -- run 2.1/2.2 first")
else:
    enriched = analysis.enrich(metrics)

    # --- corpus-level: where does each model's error go? ---------------------
    analysis.show(analysis.error_composition(enriched),
                  "Error composition per model (% of that model's error seconds)")

    # --- the eight rankings --------------------------------------------------
    for caption, table in analysis.all_rankings(metrics, n=20).items():
        analysis.show(table, caption)

In [ ]:
if len(metrics):
    # --- head-to-head --------------------------------------------------------
    analysis.show(analysis.head_to_head_summary(enriched, "der"), "Head-to-head summary (DER)")
    analysis.show(analysis.model_comparison(enriched, "der"),
                  "Per-clip model comparison -- delta > 0 means the first model is worse")

    # --- concentration: do a few clips dominate? -----------------------------
    for model in sorted(enriched.model.unique()):
        p = analysis.pareto(enriched, model, n=15)
        analysis.show(p, f"Error concentration -- {model}")
        if len(p):
            print(f"   top {len(p)} clips = {p.cumulative_share.iloc[-1]:.1%} "
                  f"of {model}'s total error seconds")

### 2.6 — Error composition chart

Stacked bars: how each model's error seconds split between false alarm, missed
speech and confusion, for the clips contributing most. A tall bar that is mostly
purple is a clustering problem; mostly blue is a detection problem. They call for
different fixes in Step 4.

In [ ]:
if len(metrics):
    import matplotlib.pyplot as plt

    top = (enriched.sort_values("error_sec", ascending=False)
           .groupby("model").head(15))
    models = sorted(top.model.unique())
    fig, axes = plt.subplots(len(models), 1, figsize=(13, 5 * len(models)), squeeze=False)

    for ax, model in zip(axes[:, 0], models):
        sub = top[top.model == model].sort_values("error_sec")
        ax.barh(sub.clip_id, sub.der_fa_sec, label="false alarm", color="#4C9F70")
        ax.barh(sub.clip_id, sub.der_miss_sec, left=sub.der_fa_sec,
                label="missed", color="#3E7CB1")
        ax.barh(sub.clip_id, sub.der_confusion_sec,
                left=sub.der_fa_sec + sub.der_miss_sec, label="confusion", color="#8B5FBF")
        ax.set_title(f"{model} -- 15 clips contributing the most error")
        ax.set_xlabel("error seconds")
        ax.tick_params(axis="y", labelsize=8)
        ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    # overlap vs single-speaker: the condition that actually separates models
    fig, ax = plt.subplots(figsize=(7, 5))
    for model in models:
        sub = enriched[(enriched.model == model) & enriched.overlap_der.notna()]
        ax.scatter(sub.single_speaker_der, sub.overlap_der, alpha=0.65, label=model)
    lim = [0, max(1.0, enriched.overlap_der.max(skipna=True) or 1.0)]
    ax.plot(lim, lim, "k--", lw=1, label="equal difficulty")
    ax.set_xlabel("DER on single-speaker regions")
    ax.set_ylabel("DER on overlapped regions")
    ax.set_title("Points above the line = overlap is harder for that clip")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2.7 — Export the standalone error explorer

Writes a self-contained `error_explorer/` you download once and open locally.
No hosted runtime, no backend, no build step, no internet.

Everything it shows was computed here and serialised — the browser does no metric
arithmetic. That is deliberate: a second implementation in JavaScript would be
free to drift from the one that produced the benchmark. The exported MISS / FA /
CONFUSION regions are **verified at export time** to reproduce pyannote's own DER
components to within a microsecond, and the result is recorded in
`data/clips.json` so the UI can warn you if it ever fails.

Audio is not bundled by default — the corpus is 1.3 GB and does not belong in a
git repo. The timeline, error regions and filtering all work without it; copy
individual WAVs into `error_explorer/audio/` for the clips you want to hear, or
pass `copy_audio=True`.

To use it:

```bash
cd error_explorer
python -m http.server 8000
# open http://localhost:8000
```

In [ ]:
from sarvam_diar import explorer

if not len(metrics):
    print("nothing to export -- run 2.1/2.2 first")
else:
    out = explorer.export(
        cfg, metrics, references, hypotheses,
        out_dir=cfg.root / "error_explorer",
        copy_audio=False,   # True bundles ~1.3 GB of WAV
        verify=True,        # assert regions reproduce the scored components
    )

    import json
    manifest = json.load(open(out / "data" / "clips.json"))
    bad = manifest["verification"]["mismatches"]
    print(f"\n{manifest['n_clips']} clips, models={manifest['models']}")
    print(f"verification mismatches: {len(bad)}"
          + ("  <-- regions would NOT match the metric, do not trust the view" if bad else "  (regions match the metric)"))

    size = sum(f.stat().st_size for f in out.rglob("*") if f.is_file())
    print(f"export size: {size/1e6:.1f} MB at {out}")
    print("\nDownload the folder, then:")
    print("    cd error_explorer && python -m http.server 8000")
    print("    open http://localhost:8000")

    # Optional: zip it so it downloads as one file from the Output tab.
    import shutil
    zip_path = shutil.make_archive(str(cfg.root / "error_explorer"), "zip", root_dir=out)
    print(f"\nzipped -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
    if IN_KAGGLE:
        print("\nThe zip is under /kaggle/working -- download it from the Output"
              "\ntab on the right after the notebook finishes, or 'Save Version'"
              "\nto persist it.")